In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.4 Cross-entropy

> 🎯 **Goal:** Understand the single number we minimize during training: how surprised the model was by the correct answer.

**Why this matters.** Training a model means improving it, and "improving" needs a precise definition of what we are improving toward. That definition is the **loss**: one number that says how wrong the model currently is. For language models the loss is **cross-entropy**, and the entire training process is just relentlessly pushing this number down. Without it, the optimizer would have no idea which direction is "better".

**The intuition.** Cross-entropy measures surprise. The model assigns a probability to the token that actually came next. If it confidently predicted the right token (high probability), it was barely surprised, and the loss is small. If it gave the right token a tiny probability, it was very surprised, and the loss is large. The negative logarithm is the tool that turns "the probability I gave the truth" into "how surprised I was": a probability of 1 gives a loss of 0 (no surprise), and a probability near 0 gives a huge loss (total shock).

**The mechanics.** Cross-entropy for a single prediction is `-log(p_correct)`, where `p_correct` is the softmax probability the model placed on the true next token. In practice we combine softmax and log into one numerically-stable step called `log_softmax`, then pick out the entry for the correct token and negate it. That is exactly what the by-hand line does. `F.cross_entropy` does the same thing in one call, and crucially it takes raw **logits** (not probabilities) plus the integer index of the correct token; it applies the softmax internally, so never softmax your logits before handing them to it. Here `target = [0]` means "the correct next token is index 0", which happens to be the highest logit, so we expect a small loss.

In [ ]:
from torch.nn import functional as F

logits = torch.tensor([[2.0, 0.5, 0.1]])
target = torch.tensor([0])
by_hand = -F.log_softmax(logits, dim=-1)[0, target.item()]
builtin = F.cross_entropy(logits, target)
print("by hand:", by_hand.item(), " F.cross_entropy:", builtin.item())

**What just happened.** Both methods printed the same small loss (around `0.46`). It is small because the model's highest logit was on index 0, which we declared correct, so the model was not very surprised. The by-hand path and `F.cross_entropy` agree, proving they are the same computation. A useful sanity anchor: a model guessing uniformly over a vocabulary of size V scores exactly `ln(V)` (for example, about `4.16` for a 64-character vocabulary). Beating that baseline is the very first measurable sign that your model is learning rather than guessing.

⚠️ **Common pitfalls**
- Applying softmax before `F.cross_entropy`. The function expects raw logits and softmaxes internally. Softmaxing first means it happens twice, quietly corrupting your loss.
- Wrong target shape or type. The target must be a `long` (integer) tensor of class indices, not one-hot vectors and not floats. A mismatch here produces confusing errors or silently wrong numbers.

🏋️ **Try it yourself.** Change `target` to `torch.tensor([2])`, declaring that the lowest logit (`0.1`) was actually correct, and rerun. The loss jumps up, because now the model was very surprised by the truth. Watching the loss rise when the prediction is wrong and fall when it is right is the whole intuition behind training.

In [ ]:
assert torch.allclose(by_hand, builtin)